# mim-alunni-corso-eta — notebook v0

Notebook v0 di validazione del mart in `dataset-incubator`.

- scopo: sanity check e lettura base del mart
- non sostituisce l'analisi pubblica
- evitare output pesanti o immagini embeddate nel commit


In [1]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

SLUG = "mim-alunni-corso-eta"
DATASET_NAME = "mim_alunni_corso_eta"
MART_TABLE = "mart_alunni_primaria_comune"
METRICA = "alunni_totali"

candidate_dir = Path.cwd()
if not (candidate_dir / "dataset.yml").exists():
    if (candidate_dir.parent / "dataset.yml").exists():
        candidate_dir = candidate_dir.parent
    else:
        raise FileNotFoundError(
            f"dataset.yml non trovato in {Path.cwd()} o nella cartella parent."
        )

mart_root = candidate_dir.parents[1] / "out" / "data" / "mart" / DATASET_NAME
mart_candidates = sorted(mart_root.glob(f"*/{MART_TABLE}.parquet"))
if not mart_candidates:
    raise FileNotFoundError(
        f"Mart non trovato per {DATASET_NAME}/{MART_TABLE} sotto {mart_root}. Eseguire prima toolkit run all."
    )

PARQUET_PATH = mart_candidates[-1]
print(f"Candidate dir: {candidate_dir.name}")
print(f"Mart table: {PARQUET_PATH.name}")


Candidate dir: mim-alunni-corso-eta
Mart table: mart_alunni_primaria_comune.parquet


In [2]:
con = duckdb.connect()
df = con.execute("SELECT * FROM read_parquet(?)", [str(PARQUET_PATH)]).df()
print(f"Shape: {df.shape}")
display(df.dtypes)


Shape: (6281, 8)


anno_scolastico             str
area_geografica             str
regione                     str
provincia                   str
codice_comune_scuola        str
comune                      str
scuole_osservate          int64
alunni_totali           float64
dtype: object

In [3]:
display(df.head(10))


,anno_scolastico,area_geografica,regione,provincia,codice_comune_scuola,comune,scuole_osservate,alunni_totali
0,202425,ISOLE,SARDEGNA,ORISTANO,B314,CABRAS,2,236.0
1,202425,SUD,CALABRIA,CATANZARO,M208,LAMEZIA TERME,16,3146.0
2,202425,ISOLE,SICILIA,MESSINA,B666,CAPO D'ORLANDO,6,546.0
3,202425,SUD,PUGLIA,BARLETTA-ANDRIA-TRANI,A883,BISCEGLIE,9,2254.0
4,202425,ISOLE,SICILIA,CATANIA,C351,CATANIA,74,13019.0
5,202425,SUD,CALABRIA,CROTONE,F157,MESORACA,3,282.0
6,202425,SUD,MOLISE,ISERNIA,A761,BELMONTE DEL SANNIO,1,8.0
7,202425,SUD,CAMPANIA,SALERNO,I483,SCAFATI,13,2237.0
8,202425,ISOLE,SICILIA,ENNA,G624,PIETRAPERZIA,3,269.0
9,202425,NORD OVEST,LOMBARDIA,SONDRIO,B255,BUGLIO IN MONTE,1,97.0


In [4]:
print("Null per colonna:")
display(df.isnull().sum())

if METRICA in df.columns:
    negativi = int((df[METRICA] < 0).sum()) if pd.api.types.is_numeric_dtype(df[METRICA]) else "n/a"
    print(
        f"\nRange {METRICA}: min={df[METRICA].min()}, max={df[METRICA].max()}, negativi={negativi}"
    )
else:
    print(f"Metrica non trovata nel mart: {METRICA}")


Null per colonna:


anno_scolastico         0
area_geografica         0
regione                 0
provincia               0
codice_comune_scuola    0
comune                  0
scuole_osservate        0
alunni_totali           0
dtype: int64


Range alunni_totali: min=1.0, max=92215.0, negativi=0


In [5]:
# DIM = "comune"
# (
#     df.groupby(DIM)[METRICA]
#     .sum()
#     .sort_values(ascending=False)
#     .plot(kind="bar", figsize=(12, 5), title=f"{METRICA} per {DIM}")
# )
# plt.tight_layout()
# plt.show()
print("Cella visualizzazione opzionale: decommentare e adattare a comune")


Cella visualizzazione opzionale: decommentare e adattare a comune


## Note v0

- Slug: `mim-alunni-corso-eta`
- Tabella mart: `mart_alunni_primaria_comune`
- Metrica guida: `alunni_totali`
- Perimetro: primarie statali, anno scolastico 2024/25, aggregazione comunale via support dataset SCUANAGRAFESTAT
- Questo notebook resta esplorativo e validativo in DI.
